<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/07_Customer_Lifetime_Value/01_Silent_Attrition_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 7: Predicting Silent Attrition & Customer Lifetime Value

## The Business Problem: "Silent Attrition"
In subscription-based businesses (like SaaS or streaming), churn is explicit: a customer actively cancels their contract. However, grocery retail is a **non-contractual** domain. Customers do not notify the store when they decide to shop at a competitor; they simply stop showing up. This is known as "Silent Attrition."

Furthermore, measuring this absence requires deep personalization:
* If a **Power Shopper** (who historically visits every 3 days) is absent for 14 days, there is a high probability they have churned.
* If a **Premium Shopper** (who historically visits once a month for specialty items) is absent for 14 days, their behavior is completely normal.

Traditional binary churn models fail because they apply a universal time threshold to all users.

## The Solution: "Buy 'Til You Die" (BTYD)
To solve this, we implement the **BG/NBD (Beta Geometric / Negative Binomial Distribution)** statistical framework. Instead of predicting a binary "Churned vs. Active" state, this probabilistic model calculates the unique "heartbeat" of every individual shopper.

It evaluates customers based on three factors:
1. **Recency:** The age of the customer at their last purchase.
2. **Frequency:** The number of repeat purchases they have made.
3. **Monetary Value:** Their average basket size.

## Project Execution Steps
* **Step 1: RFM Calibration:** Compress 1.4 million transactional scans into a standardized Recency, Frequency, and Monetary (RFM) matrix.
* **Step 2: Probability Modeling:** Train the BG/NBD model to calculate $P(Alive)$—the mathematical probability that a specific customer is still an active shopper.
* **Step 3: Persona Intervention:** Cross-reference high-flight-risk customers with our existing segmentation clusters to identify exactly *which* types of shoppers the business is losing.

## Step 1: RFM Calibration (Recency, Frequency, Monetary)

To feed our data into the BG/NBD probability model, we must transform the raw transactional log into a specialized summary matrix.

Using the `lifetimes` library, we compress the chronological receipt data into four mathematical vectors per customer:
* **Frequency ($x$):** Count of repeat shopping days.
* **Recency ($t_x$):** Time elapsed between the first and last purchase.
* **Age ($T$):** Total time observed from the first purchase to the end of the dataset.
* **Monetary Value ($m$):** Average spend across repeat purchases.

In [2]:
!pip install completejourney_py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 58.5 MB/s eta 0:00:00


In [3]:
# 1. Install the BTYD modeling library
!pip install lifetimes

import pandas as pd
from completejourney_py import get_data
from lifetimes.utils import summary_data_from_transaction_data

# 2. Fetch the raw transactions
print("Fetching transaction data...")
data = get_data()
transactions = data["transactions"]

# 3. Clean the date formatting
transactions['transaction_timestamp'] = pd.to_datetime(transactions['transaction_timestamp'])
transactions['date'] = transactions['transaction_timestamp'].dt.date

# 4. Generate the BTYD Summary Matrix
# We compress the data by household_id, grouping by day.
print("Calibrating RFM Matrix...")
rfm_matrix = summary_data_from_transaction_data(
    transactions,
    customer_id_col='household_id',
    datetime_col='date',
    monetary_value_col='sales_value',
    freq='D' # Daily frequency evaluation
)

print(f"✅ Matrix built! We have calibrated {len(rfm_matrix)} unique households.")
display(rfm_matrix.head(10))

Fetching transaction data...
Calibrating RFM Matrix...
✅ Matrix built! We have calibrated 2469 unique households.


,frequency,recency,T,monetary_value
household_id,,,,
1,46.0,358.0,359.0,50.908478
2,19.0,331.0,359.0,52.668947
3,19.0,349.0,359.0,53.153158
4,17.0,339.0,361.0,24.387647
5,16.0,289.0,349.0,18.023750
6,135.0,361.0,361.0,25.546148
7,31.0,344.0,349.0,62.280000
8,58.0,359.0,362.0,51.942931
9,10.0,349.0,353.0,61.785000


#Explanation:
Looking at the extremes in our output:

Household 1: Made 46 repeat purchases. Their first and last purchase were 358 days apart ($Recency$), and they have been a known customer for 359 days ($Age$). This means they shopped yesterday. They are highly active.

Household 10: Made 0 repeat purchases. They bought once, 152 days ago, and never came back. In retail, this is known as a "One-and-Done" shopper.

Now that we have the math, we can bring in the BG/NBD Model. This algorithm learns the dropout rate of the entire population and uses it to calculate the exact probability that any specific customer is still "alive."